## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e4

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E4" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E4/smoke" if SMOKE else "E4"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e4.build_initial_spec(SMOKE)


## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e4.E4_MODULES, completion_keys=e4.COMPLETION_KEYS)
e4.verify_spec_immutable(OUT, SPEC, e4.E4_MODULES)


## 3. Leakage test

In [ ]:
leaks = e4.run_leakage_e4()
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
print(ldf.to_string(index=False))
assert ldf["pass"].all(), ldf[~ldf["pass"]]


## 4. Load data and freeze universes

In [ ]:
DATA = e4.load_all(SPEC, OUT)
e4.verify_spec_immutable(OUT, SPEC, e4.E4_MODULES)
print(SPEC["universes"])
print("OM sha256:", SPEC["om_sha256"])


## 5. Variance series and returns

In [ ]:
for tier in ("OM", "A", "BN"):
    print(tier, {a: len(s["dates"]) for a, s in DATA[tier]["assets"].items()})


## 6. Fixed-model forecasts and losses

## 7. Features

## 8. Walk-forward loss predictor

## 9. Primary decision

In [ ]:
RESULT = e4.run_pipeline(SPEC, OUT, FIG, CACHE, data=DATA)
print(RESULT["primary"])


## 10. Secondary

In [ ]:
print(RESULT["secondary"].to_string(index=False))


## 11. Save artefacts

In [ ]:
print(sorted(p.name for p in OUT.glob("*.parquet")))


## 12. Figures and tables

In [ ]:
print(pd.read_csv(OUT / "decision.csv").to_string(index=False))
print(pd.read_csv(OUT / "results_primary.csv").to_string(index=False))
print(pd.read_csv(OUT / "per_asset.csv").to_string(index=False))
